In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
import pandas as pd
import os

In [ ]:
# --- Portable paths (replaces machine-specific os.chdir) ---------------
# Resolve the deposit root from wherever the notebook is launched so it
# runs anywhere. Intermediate/derived files are written to code/outputs/.
from pathlib import Path
_cwd = Path.cwd()
REPO = next((p for p in [_cwd, *_cwd.parents]
             if (p / 'construct').is_dir() and (p / 'examples').is_dir()), _cwd)
OUTDIR = REPO / 'code' / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Deposit root:', REPO)
print('Outputs ->', OUTDIR)

## ++Load plasmid and sequence++
Change the path for csv_file to a the path with a CSV file that has the following indormation regarding  the insert:
>- Name: name of peptide
>- peptide: Peptide sequence
>- FW: Forward assembly primer
>- RV: Reverse assembly primer
>- hr: Homology region 
>- length_hr:length of homology region
>- Seq_length: Sequence length 
It is important that the sequence are set to lowercase to make hte rest of the code case-insensitive

NOTE: Path to the fasta will very likely always be that one, unless the plasmid used is changed.
It will print out the last 25 nucleotides of the plasmid and the first forward primer

In [ ]:
# Load plasmid DNA sequence from FASTA file
fasta_file = str(REPO / 'construct' / 'YFB001_pET24b_His6-SUMO-linker-ccdB.fasta')
with open(fasta_file, 'r') as file:
    plasmid_record = SeqIO.read(file, 'fasta')
    plasmid_sequence = str(plasmid_record.seq).lower()  # Convert to lowercase
    
# Load primer sequences from CSV file
csv_file = str(OUTDIR / 'pcr_input.csv')  # produced by 1_primer_design.ipynb
df = pd.read_csv(csv_file)

# Ensure all primer sequences are strings and convert to lowercase
df['Forward'] = df['Forward'].astype(str).str.lower()
df['Reverse'] = df['Reverse'].astype(str).str.lower()
df['hr'] = df['hr'].astype(str).str.lower()

print("plasmid:", plasmid_sequence[:25] + '...')  
print("Forward primer:", df['Forward'].head(1)) 

# Prepare the sequence for the assembly PCR -- this includes:
> 1) Defining binding sites for the corresponding plasmids (this only changes when working with different plasmids)
> 2) Reindexing the plasmid so that the Forward Primer is 5' from the Reverse: Makes things a lot easier this way!
> 3) Print a check to verify that the binding sites are found and that the plasimd reindexing is happening

In [ ]:
## Defining primers' binding sites ##
# Corresponds to the sequence in the primer that will bind the plasmid
fw_binding_site = 'tgttgacgctgagccggagt'  # FW primer/vector adapter (public construct backbone) 
rv_binding_site = 'acaacctcccgagctgcct'   # REV primer/vector adapter (public construct backbone)  

# Specifying the binding site in the plasmid that will bind to primers - this is not the same as the sequence for the reverse primers
rv_binding_site_rc = str(Seq(rv_binding_site).reverse_complement())

# Reindex the plasmid sequence so that the forward primer binding site is 5' from the reverse primer.
def reindex_sequence(sequence, binding_site):
    site_position = sequence.find(binding_site)
    if site_position == -1:
        return None, -1  # Binding site not found
    reindexed_sequence = sequence[site_position:] + sequence[:site_position]
    return reindexed_sequence, site_position

reindexed_sequence, fw_site = reindex_sequence(plasmid_sequence, fw_binding_site)

# Verify the reindexed sequence
if reindexed_sequence:
    print(f"Reindexed Sequence start: {reindexed_sequence[:25]}...")  # Print the last 50 bases of the reindexed sequence
else:
    print("Forward binding site not found in the plasmid sequence.")
    
# Confirm the binding sites are in the plasmid sequence and verify the position where they are
fw_site = reindexed_sequence.find(fw_binding_site)
rv_site = reindexed_sequence.find(rv_binding_site_rc)

df['plasmid'] = reindexed_sequence
df['fw_binding_index'] = fw_site
df['rev_binding_index'] = rv_site

print(f"Forward binding site position: {fw_site}","| "  f"Reverse binding siteposition: {rv_site}")


## Defining primers extension sites for the plasmid:
- here it removes the primer binding site (which is the same for all primers targeting the same plasmid) to have prepare the sequences that need to be added the plasmid sequence in the PCR simulation.

In [ ]:
# Function to remove all occurrences of binding site <<from primer sequence >> 
def remove_binding_site(primer, binding_site):
    return primer.replace(binding_site, '') 

# Separate binding sites and extensions from the primers
df['fw_extension'] = df['Forward'].apply(lambda x: remove_binding_site(x, fw_binding_site))
df['rv_extension'] = df['Reverse'].apply(lambda x: remove_binding_site(x, rv_binding_site))
fw_extension = df['fw_extension']
rv_extension = df['rv_extension']

print(df[['fw_extension', 'rv_extension']].head(1))

## This Section creates the PCR product = Primers (Extension + Primers) + plasmid sequence
 1. This simulates the output of the qPCR performed 
 2. Prints and saves to a csv the linear PCR product indexed from the 5' of Forward primer

In [ ]:
# Function to simulate PCR product using the reindexed plasmid sequence
def simulate_amplicon(reindexed_sequence, fw_binding_site, rv_binding_site_rc, fw_extension, rv_extension, max_prints=3):
    # Find the reverse binding site in the reindexed sequence
    rv_site = reindexed_sequence.find(rv_binding_site_rc)
    
    # Extract the plasmid segment between the forward and reverse binding sites
    plasmid_segment = reindexed_sequence[len(fw_binding_site):rv_site]
    rv_extension_rc = str(Seq(rv_extension).reverse_complement())
    
    # Generate the PCR product sequence
    amplicon = fw_extension + fw_binding_site + plasmid_segment + rv_binding_site_rc + rv_extension_rc
    return amplicon

## Simulate PCR products for each pair of primers using reindexed sequence and calculate their lengths
amplicon_lengths = []
amplicons = []
max_prints = 1  # Maximum number of times to print the reverse binding site position
for index, row in df.iterrows():
    amplicon_sequence = simulate_amplicon(reindexed_sequence, fw_binding_site, rv_binding_site_rc, row['fw_extension'], row['rv_extension'], max_prints=max_prints)
    if amplicon_sequence:
        amplicon_lengths.append(len(amplicon_sequence))
        amplicons.append(amplicon_sequence)
    else:
        amplicon_lengths.append(None)
        amplicons.append(None)

# Add the lengths of the PCR products as a new column in the DataFrame
df['amplicon_length'] = amplicon_lengths
df['amplicon_sequence'] = amplicons

print(df[['name', 'amplicon_length']].head(2))


## ++This section simulates the in vivo assembly ++ 
 1. Notes homology regions lengths and using this it substract one homology region and joins the plasmid before and after it
 2. Prints out the circularised products and creates 2 new columns: cloned_amplicons and cloned_len

In [ ]:
 # Function to clone (circularize) the amplicon
def clone(amplicon, homology_length):
    # Extract the homology regions from the 5' and 3' ends
    homology_5 = amplicon[:homology_length]
    homology_3 = amplicon[-homology_length:]
    
    # Verify homology
    if homology_5 != homology_3:
        print("Warning: Homology regions do not match!")
    
    # Circularize the product by removing one of the homology regions
    cloned_amplicon = amplicon[:-homology_length]  # Remove only the homology region from the 3' end
    
    return cloned_amplicon

# Circularize PCR products
cloned_amplicons = []
for index, row in df.iterrows():
    homology_length = int(df.loc[index, 'length_hr'])
    amplicon_sequence = row['amplicon_sequence']
    if amplicon_sequence:
        cloned_sequence = clone(amplicon_sequence, homology_length)
        cloned_amplicons.append(cloned_sequence)
    else:
        cloned_amplicons.append(None)

df['cloned_amplicon'] = cloned_amplicons
df['cloned_len'] = df['cloned_amplicon'].apply(lambda x: len(x) if x else None)

print(df[['name','length_hr','amplicon_length', 'cloned_len']].head(5))

## +++ This is to define the Open Reading Frame (ORF)togenerate the translated product +++
# To do this it will: 
    1. Define the 'T7 promoter sequence' 
    2. Reindexing the plasmid to start at the T7 promoter
    3. It will look for the 1st 'RBS sequence' (after the T7 promoter - indexewd earlier)
    4. It will look for the first Start codon (ATG) after the RBS
    5. It will look for a stop codon on the form of TAA, TAG, or TGA in hte same reading frame as the start codon.

> The sequences wil not change for this plasmid, but will if you chnage plasmid you need to update the sequences 
    >> The seq given for T7_promoter is the one in most pET plasmids
    >> the reindex_sequence is the sequence exactly 5' of the promoter

In [ ]:
# Define the T7 promoter and the sequence to reindex by
t7_promoter = 'taatacgactcactataggg'
reindex_sequence = 'gtccggcgtagaggatcgagatctcgatcccgcgaaat'  # 38 nt immediately 5' of T7 in the construct

# Function to reindex the sequence to start just before the T7 promoter
def reindex_promoter(sequence, reindex_sequence):
    # Find the position of the reindex sequence
    reindex_position = sequence.find(reindex_sequence)
    
    if reindex_position == -1:
        print("Reindex sequence not found in the circularized product.")
        return sequence
    
    # Calculate the new starting position (just after the reindex sequence)
    new_start = reindex_position + len(reindex_sequence)
    
    # Reindex the sequence
    reindexed_clone = sequence[new_start:] + sequence[:new_start]
    
    return reindexed_clone

# Reindex each circularized product to start before the T7 promoter
reindexed_clones = []
for index, row in df.iterrows():
    cloned_amplicon = row['cloned_amplicon']
    reindexed_clone = reindex_promoter(cloned_amplicon, reindex_sequence)
    reindexed_clones.append(reindexed_clone)

# Add the reindexed circularized sequences to the original DataFrame
df['reindexed_clones'] = reindexed_clones

# Deinfining and finding the RBS, Start and Stop codons:
def find_ORF(sequence, rbs, start_codon='atg', stop_codons=('taa', 'tag', 'tga')):
    # Find the RBS position 
    rbs_position = sequence.find(rbs)
    if rbs_position == -1:
        print(f"RBS '{rbs}' not found in the sequence.")
        return None
    
    # Searching for the start codon immediately after the RBS
    start_search_position = rbs_position + len(rbs)
    start_codon_position = sequence.find(start_codon, start_search_position)
    if start_codon_position == -1:
        print(f"Start codon '{start_codon}' not found after the RBS.")
        return None

    # Find the first stop codon (in the same reading frame as the start codon)
    stop_codon_position = -1
    for i in range(start_codon_position + 3, len(sequence) - 2, 3):
        codon = sequence[i:i + 3]
        if codon in stop_codons:
            stop_codon_position = i
            break

    if stop_codon_position == -1:
        print("No stop codon found after the start codon.")
        return None
    
    # Extract the ORF
    ORF = sequence[start_codon_position:stop_codon_position + 3]
    return ORF

# Define the RBS sequence
rbs_sequence = "tttgtttaactttaagaaggaga"

# Find ORF for each circularized product and add it to the DataFrame
ORFs = []
ORF_lengths = []

for index, row in df.iterrows():
    reindexed_clones = row['reindexed_clones']
    ORF = find_ORF(reindexed_clones, rbs_sequence)
    ORFs.append(ORF)
    ORF_lengths.append(len(ORF) if ORF else None)

df['ORF'] = ORFs
df['ORF_length'] = ORF_lengths

print(df[['ORF', 'ORF_length']].head(3))

# Creating the AA sequence from the ORF
 > this wil use Bio.Seq function to generate the AA sequence

In [ ]:
from Bio.Seq import Seq

# Function to translate ORF to amino acid sequence
def translate_orf(orf):
    if orf is None:
        return None
    # Create a Seq object and translate it to amino acids
    seq_obj = Seq(orf)
    amino_acid_seq = seq_obj.translate()
    return str(amino_acid_seq)

# Translate the ORF for each circularized product and add it to the DataFrame
tagged_peptides = []
peptide_lengths = []
for index, row in df.iterrows():
    orf = row['ORF']
    peptide = translate_orf(orf)
    tagged_peptides.append(peptide)
    if peptide:
        peptide_lengths.append(len(peptide))
    else:
        peptide_lengths.append(None)

# Add the tagged peptides and their lengths to the DataFrame
df['tagged_peptide'] = tagged_peptides
df['tagged_peptide_length'] = peptide_lengths

# Print the first few rows to verify the results
print(df[['ORF', 'tagged_peptide', 'tagged_peptide_length']].head(2))

### ++ Save into a SEC input file ++
> Preparing for analysis as required in the SEC dspep workflow.

In [ ]:
# Save the PCR_output.csv file
pcr_output_path = str(OUTDIR / 'PCR_output.csv')
df.to_csv(pcr_output_path, index=False)

# Define SEC info
sec_df = df[['Number', 'Source Plate Name', 'Well', 'name', 'Peptide','eblock','ORF','tagged_peptide','Peptide_length','tagged_peptide_length']]

# Save the extracted columns to a new CSV file
sec_input_path = str(OUTDIR / 'sec_input.csv')  # input to 3_sec_analysis.ipynb
sec_df.to_csv(sec_input_path, index=False)

print(f"Selected columns saved to {sec_input_path}")
print(f"cloning and expression data saved to {pcr_output_path}")

In [ ]:
sec_df